# Lesson 02: Thresholds and Uncertainty

A model score is not yet a policy. This lesson sweeps decision thresholds and separates the calibration set used to choose a candidate from the held-out set used to report it. The included data is synthetic until you supply a real, labeled run. Never tune on the held-out test set.

In [ ]:
from jev_eval.metrics import binary_metrics

# Clearly synthetic: replace with scores from a calibration dataset.
calibration_labels = [True, True, False, False, True, False]
calibration_scores = [0.92, 0.41, 0.62, 0.08, 0.77, 0.25]
test_labels = [True, False, True, False]
test_scores = [0.69, 0.18, 0.31, 0.83]
print("Synthetic demonstration only")

## Sweep on calibration data

Choose a policy using an explicitly stated constraint, such as maximizing caught attacks while holding the benign false-block rate below a chosen tolerance. With tiny samples, the apparent optimum is unstable; use confidence intervals and adequate sample sizes before relying on it.

In [ ]:
max_false_block_rate = 0.10
candidates = []
for threshold_step in range(5, 96, 5):
    threshold = threshold_step / 100
    metrics = binary_metrics(calibration_labels, calibration_scores, threshold=threshold)
    if metrics["false_block_rate"] is not None and metrics["false_block_rate"] <= max_false_block_rate:
        candidates.append((metrics["recall"], threshold, metrics))

best = max(candidates, key=lambda item: (item[0], item[1])) if candidates else None
if best:
    selected_threshold = best[1]
    print({"calibration_threshold": selected_threshold, "calibration_metrics": best[2]})
else:
    selected_threshold = 0.5
    print("No threshold met the chosen constraint; using 0.5 for illustration.")

## Evaluate once on held-out data

The following is only a split-discipline demonstration because its sample scores are synthetic. In a real run, freeze the selected threshold before opening the held-out results. Report misses and false blocks, not just accuracy.

In [ ]:
held_out_metrics = binary_metrics(test_labels, test_scores, threshold=selected_threshold)
print({"held_out_threshold": selected_threshold, "held_out_metrics_synthetic": held_out_metrics})